# 01 — Exploratory Data Analysis

**Project:** Early Prediction of Urban Air-Quality Episodes

Inspect the Beijing Multi-Site Air-Quality dataset (UCI #501) for the configured primary site.

Prerequisite: `python scripts/download_data.py`

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config, project_path
from src.data.loader import download_dataset, load_site, dataset_summary
from src.data.validation import validate_raw_frame, print_validation_report
from src.data.preprocessing import preprocess_site
from src.features.targets import create_target_from_config, target_prevalence

sns.set_theme(style="whitegrid", context="notebook")
cfg = load_config()
download_dataset()
raw = load_site()
report = validate_raw_frame(raw)
print_validation_report(report)
dataset_summary(raw)

In [ ]:
cols = [c for c in ["PM2.5","PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM"] if c in raw.columns]
miss = raw[cols].isna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8,4))
miss.plot(kind="bar", ax=ax)
ax.set_ylabel("Fraction missing"); ax.set_title("Missingness by variable")
plt.tight_layout()
plt.savefig(project_path("reports/figures/eda_missingness.png"), dpi=150)
plt.show()
miss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(raw["PM2.5"].dropna(), bins=60, ax=axes[0])
axes[0].axvline(cfg["target"]["threshold_ug_m3"], color="crimson", ls="--", label="threshold")
axes[0].legend(); axes[0].set_title("PM2.5 histogram")
sns.boxplot(x=raw["PM2.5"].dropna(), ax=axes[1])
axes[1].set_title("PM2.5 boxplot")
plt.tight_layout()
plt.savefig(project_path("reports/figures/eda_pm25_distribution.png"), dpi=150)
plt.show()

In [ ]:
tmp = raw.set_index("timestamp")["PM2.5"].resample("D").mean()
fig, ax = plt.subplots(figsize=(12,4))
tmp.plot(ax=ax)
ax.axhline(cfg["target"]["threshold_ug_m3"], color="crimson", ls="--")
ax.set_title("Daily mean PM2.5"); ax.set_ylabel("µg/m³")
plt.tight_layout()
plt.savefig(project_path("reports/figures/eda_pm25_daily_trend.png"), dpi=150)
plt.show()

In [ ]:
df = raw.copy()
df["month"] = df["timestamp"].dt.month
df["hour"] = df["timestamp"].dt.hour
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.boxplot(data=df, x="month", y="PM2.5", ax=axes[0], showfliers=False)
axes[0].set_title("PM2.5 by month")
sns.lineplot(data=df, x="hour", y="PM2.5", ax=axes[1], errorbar=("ci", 95))
axes[1].set_title("PM2.5 diurnal cycle")
plt.tight_layout()
plt.savefig(project_path("reports/figures/eda_seasonal_hourly.png"), dpi=150)
plt.show()

In [ ]:
corr = raw[cols].corr()
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Pearson correlation")
plt.tight_layout()
plt.savefig(project_path("reports/figures/eda_correlation.png"), dpi=150)
plt.show()

In [ ]:
clean = preprocess_site(raw, cfg)
labeled = create_target_from_config(clean, cfg)
print(target_prevalence(labeled["y_episode"]))
fig, ax = plt.subplots(figsize=(5,4))
labeled["y_episode"].dropna().astype(int).value_counts().sort_index().plot(
    kind="bar", ax=ax, color=["#54A24B","#E45756"]
)
ax.set_xticklabels(["No episode (0)", "Episode (1)"], rotation=0)
ax.set_title("Next-24h elevated episode labels")
plt.tight_layout()
plt.savefig(project_path("reports/figures/eda_episode_balance.png"), dpi=150)
plt.show()